In [1]:
import gensim

# Load FastText embeddings
def load_embeddings(file_path):
    return gensim.models.KeyedVectors.load_word2vec_format(file_path)

# Load English and Hindi embeddings
en_embeddings = load_embeddings(r'C:\Users\hseng\OneDrive\Desktop\MLE_sarvam\vec_models\cc.en.300.vec.gz')
hi_embeddings = load_embeddings(r'C:\Users\hseng\OneDrive\Desktop\MLE_sarvam\vec_models\cc.hi.300.vec.gz')

# Check embedding size and words
print(f'English embedding size: {en_embeddings.vector_size}, Hindi embedding size: {hi_embeddings.vector_size}')
print(f'English vocabulary size: {len(en_embeddings)}, Hindi vocabulary size: {len(hi_embeddings)}')


English embedding size: 300, Hindi embedding size: 300
English vocabulary size: 2000000, Hindi vocabulary size: 1876653


In [36]:
# select top 10000 words from each language
en_words = list(en_embeddings.index_to_key)[:100000]
hi_words = list(hi_embeddings.index_to_key)[:100000]

In [37]:
import torch.nn as nn

# Define Discriminator as per the paper
class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# Initialize Discriminator
discriminator = Discriminator(input_dim=300, hidden_dim=2048)
print(discriminator)


Discriminator(
  (model): Sequential(
    (0): Linear(in_features=300, out_features=2048, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=2048, out_features=2048, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=2048, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [38]:
# Define Generator as the mapping matrix W
class Generator(nn.Module):
    def __init__(self, input_dim):
        super(Generator, self).__init__()
        # Initialize the mapping matrix W as a linear transformation
        self.mapping = nn.Linear(input_dim, input_dim, bias=False)

    def forward(self, x):
        return self.mapping(x)

# Initialize Generator
generator = Generator(input_dim=300)
print(generator)


Generator(
  (mapping): Linear(in_features=300, out_features=300, bias=False)
)


In [39]:
import numpy as np



import torch

# Helper function to sample random embeddings from FastText
def get_random_embeddings(embedding_model, num_samples):
    words = np.random.choice(list(embedding_model.key_to_index.keys()), num_samples)
    embeddings = torch.tensor([embedding_model[word] for word in words], dtype=torch.float32)
    return embeddings



In [28]:
# Initialize beta
beta = 0.01
import torch.optim as optim

# Initialize the optimizers
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.001)
g_optimizer = optim.Adam(generator.parameters(), lr=0.001)

# Training loop parameters
epochs = 10  # Number of training epochs
batch_size = 64 

# Training loop
for epoch in range(epochs):
    # Sample random embeddings from both source (English) and target (Hindi)
    source_batch = get_random_embeddings(en_embeddings, batch_size)
    target_batch = get_random_embeddings(hi_embeddings, batch_size)

    # Forward pass: Map the source embeddings to the target space using the generator
    mapped_source = generator(source_batch)

    # --- Train the Discriminator ---
    d_optimizer.zero_grad()  # Zero the gradients for the discriminator

    # Discriminator output for real target embeddings
    real_preds = discriminator(target_batch)  
    # Discriminator output for fake mapped source embeddings
    fake_preds = discriminator(mapped_source.detach())  

    # Calculate discriminator losses
    real_loss = nn.BCELoss()(real_preds, torch.ones_like(real_preds))  # Real labels = 1
    fake_loss = nn.BCELoss()(fake_preds, torch.zeros_like(fake_preds))  # Fake labels = 0

    # Total discriminator loss
    d_loss = real_loss + fake_loss

    # Backpropagation and optimization for the discriminator
    d_loss.backward()  # Compute gradients
    d_optimizer.step()  # Update discriminator weights

    # --- Train the Generator ---
    g_optimizer.zero_grad()  # Zero the gradients for the generator
    
    # Recompute discriminator output for mapped source embeddings
    fake_preds = discriminator(mapped_source)

    # Generator loss: fool the discriminator to predict 1 (real)
    g_loss = nn.BCELoss()(fake_preds, torch.ones_like(fake_preds))

    # Backpropagation and optimization for the generator
    g_loss.backward()  # Compute gradients
    g_optimizer.step()  # Update generator weights

    # --- Update W using the specified rule ---
    with torch.no_grad():  # No gradient tracking for this operation
        W = generator.mapping.weight.data  # Get current weight matrix W
        generator.mapping.weight.data = (1 + beta) * W - beta * (W @ W.t()) @ W  # Apply update rule

    # Print loss values every 100 epochs
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, D Loss: {d_loss.item()}, G Loss: {g_loss.item()}')


Epoch 0, D Loss: 1.3850109577178955, G Loss: 0.7625113725662231


In [13]:
def load_bilingual_dictionary(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        pairs = [line.strip().split() for line in f if line.strip()]
    return pairs

# Load the bilingual dictionary (e.g., 'muse_dictionary.txt')
bilingual_pairs = load_bilingual_dictionary(r'C:\Users\hseng\OneDrive\Desktop\MLE_sarvam\english-hindi-bilingual-lexicon.txt')
print(f'Loaded {len(bilingual_pairs)} bilingual word pairs.')


Loaded 8704 bilingual word pairs.


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

def find_nearest_neighbors(source_embeddings, target_embeddings, k=3):
    # Compute cosine similarity between source and target embeddings
    similarities = cosine_similarity(source_embeddings, target_embeddings)
    
    # Get the indices of the top k nearest neighbors
    nearest_neighbors = similarities.argsort(axis=1)[:, -k:][:, ::-1]  # Get top k indices in descending order
    return nearest_neighbors


def get_embeddings_safe(embedding_model, words):
    embeddings = []
    missing_words = []
    for word in words:
        try:
            embeddings.append(embedding_model[word])
        except KeyError:
            missing_words.append(word)  # Track missing words
            embeddings.append(np.zeros(embedding_model.vector_size))  # Append a zero vector or handle it differently
    return torch.tensor(embeddings, dtype=torch.float32), missing_words

# Get embeddings for the source and target words safely
source_embeddings, missing_source_words = get_embeddings_safe(en_embeddings, source_words)
target_embeddings, missing_target_words = get_embeddings_safe(hi_embeddings, target_words)

# Print out the missing words for debugging
if missing_source_words:
    print(f'Missing source words: {missing_source_words}')
if missing_target_words:
    print(f'Missing target words: {missing_target_words}')

# Find nearest neighbors
k = 3  # Number of nearest neighbors
nearest_neighbors = find_nearest_neighbors(source_embeddings.numpy(), target_embeddings.numpy(), k)


NameError: name 'source_words' is not defined

In [16]:
def calculate_precision(nearest_neighbors, correct_indices):
    precision = {}
    total_queries = nearest_neighbors.shape[0]
    
    for k in range(1, 4):  # For Precision@1, Precision@2, Precision@3
        correct_count = 0
        for i in range(total_queries):
            if correct_indices[i] in nearest_neighbors[i, :k]:  # Check if the correct index is in the top-k neighbors
                correct_count += 1
        precision[f'Precision@{k}'] = correct_count / total_queries

    return precision

# Create a mapping from target words to their indices for correct lookups
target_word_to_index = {word: idx for idx, word in enumerate(target_words)}

# Find correct indices based on the nearest neighbors
correct_indices = [target_word_to_index[pair[1]] for pair in bilingual_pairs]

# Calculate precision
precision_scores = calculate_precision(nearest_neighbors, correct_indices)

# Print precision results
for k, score in precision_scores.items():
    print(f'{k}: {score:.4f}')


NameError: name 'target_words' is not defined

In [35]:
source_word = 'for'

# Get the embedding for the source word
source_embedding = en_embeddings[source_word]

# Map the source embedding to the target space using the generator
mapped_source_embedding = generator.mapping(torch.tensor(source_embedding, dtype=torch.float32))

# Unsqueeze to add a batch dimension
mapped_source_embedding = mapped_source_embedding.unsqueeze(0)

# Detach the tensor before converting to NumPy
mapped_source_embedding_np = mapped_source_embedding.detach().numpy()

# Find the nearest neighbors in the target space
nearest_neighbors = find_nearest_neighbors(mapped_source_embedding_np, hi_embeddings.vectors, k=5)

# Get the nearest Hindi word
nearest_hindi_word = hi_embeddings.index_to_key[nearest_neighbors[0][0]]
nearest_hindi_word_1 = hi_embeddings.index_to_key[nearest_neighbors[0][1]]
nearest_hindi_word_2 = hi_embeddings.index_to_key[nearest_neighbors[0][2]]
nearest_hindi_word_3 = hi_embeddings.index_to_key[nearest_neighbors[0][3]]
nearest_hindi_word_4 = hi_embeddings.index_to_key[nearest_neighbors[0][4]]
print(f'The nearest Hindi word for "{source_word}" is "{nearest_hindi_word}"')
print(f'The nearest Hindi word for "{source_word}" is "{nearest_hindi_word_1}"')
print(f'The nearest Hindi word for "{source_word}" is "{nearest_hindi_word_2}"')
print(f'The nearest Hindi word for "{source_word}" is "{nearest_hindi_word_3}"')
print(f'The nearest Hindi word for "{source_word}" is "{nearest_hindi_word_4}"')

The nearest Hindi word for "for" is "ओपिन"
The nearest Hindi word for "for" is "फ़ूंक"
The nearest Hindi word for "for" is "फुलाव"
The nearest Hindi word for "for" is "सकदर"
The nearest Hindi word for "for" is "दबिश"
